# 01 · Reproject input against pyproj

**Question.** Does Fieldwork's `reproject` widget, which converts WGS84 UTM easting/northing to CRS84 longitude/latitude with proj4, agree with PROJ through `pyproj`?

**Tolerance, stated before running: 1 mm on the ground.** Separation is measured as a geodesic distance in metres, not as a difference in degrees, because a degree is not a distance.

**Independence.** Nothing here imports Fieldwork code. The fixture supplies the projected input, the declared parameters and Fieldwork's output; this notebook rebuilds the transform from the declared proj4 definition and recomputes. Fieldwork's intermediate values are deliberately absent from the fixture.

Run `00-kernel-check.ipynb` first and keep its output with any result from this notebook. The design record is `docs/experiments/41-validation-lab.md` in the Fieldwork repository.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
NEEDED = ["pyproj", "shapely", "rasterio", "geopandas", "fiona", "numpy", "pandas", "h3", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    print("Falling back to piplite for anything still missing.")
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_01_reproject import run, TOLERANCE_M

report = run(Path.cwd() / "fixtures" / "reproject-utm35s-001.json")
print(json.dumps({k: report[k] for k in ["environment", "fixture", "toleranceM", "points", "maxSeparationM", "agrees"]}, indent=2))

The fixture's SHA-256 is verified before anything is compared. A mismatch stops the notebook: a result computed against an unidentified fixture is not evidence.

In [ ]:
for row in sorted(report["rows"], key=lambda r: -r["separationM"]):
    print(f"{row['id']:>4}  reference {row['reference']['longitude']:.9f}, {row['reference']['latitude']:.9f}"
          f"   fieldwork {row['fieldwork']['longitude']:.9f}, {row['fieldwork']['latitude']:.9f}"
          f"   separation {row['separationM'] * 1e6:8.1f} µm")

## Reading the residual

A nonzero separation is expected and is not evidence of a projection disagreement. Fieldwork rounds stored coordinates to **nine decimal places**, recorded as `selfReported.coordinateDecimals` in the fixture. One unit in the ninth decimal is about 1.11 × 10⁻⁴ m of latitude, so rounding alone can displace a point by up to roughly 7.5 × 10⁻⁵ m once longitude and latitude are combined. A residual at or below that scale is attributable to the documented rounding rather than to the transformation.

The cell below makes that comparison explicit instead of leaving it to the reader.

In [ ]:
import math

decimals = report["fixture"]["selfReportedMaxRoundTripM"], 9
half_unit_deg = 0.5 * 10 ** -9
metres_per_degree_lat = 111320.0
worst = max(report["rows"], key=lambda r: r["separationM"])
lat = math.radians(worst["reference"]["latitude"])
bound = math.hypot(half_unit_deg * metres_per_degree_lat,
                   half_unit_deg * metres_per_degree_lat * math.cos(lat))
print(f"observed maximum separation  {report['maxSeparationM']:.3e} m")
print(f"rounding bound at that point {bound:.3e} m")
print("explained by rounding" if report["maxSeparationM"] <= bound else
      "NOT explained by rounding: investigate the transformation")

## What this check establishes, and what it does not

It establishes that two independent implementations of the same UTM inverse projection on the WGS84 datum agree, over this fixture's spread of the zone, to the stated tolerance.

It does not establish that the declared zone or hemisphere is the right one for any given file — the same easting and northing are valid in both hemispheres, so a misdeclaration converts cleanly and relocates records between continents. It does not establish that the source coordinates were recorded correctly. It covers no datum other than WGS84, no zone other than the fixture's, and no raster warping. Agreement here is a numerical fact, not a statement about the suitability of any public-health analysis.